# Sparse Memory Finetuning — Full Pipeline (v3_collab)\n\nRuns all 3 stages on a Colab T4 GPU:\n1. **Recovery** — teach memory layers to match base model behaviour\n2. **Background stats** — collect KL slot statistics on general data\n3. **Sparse continual finetuning** — learn new knowledge into selected memory slots\n\n**What you need to upload (Cell 3):**\n- `data/oasst1_train_4000_messages.jsonl` — used by background stats\n- `data/recovery_eval_sample.jsonl` (tiny)\n- `data/recovery_prompts.json` (tiny)\n\nAll model weights and training data are downloaded automatically.

## Cell 1 — Verify GPU

In [ ]:
!nvidia-smi
import torch
print('CUDA:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## Cell 2 — Clone repo & install

In [ ]:
import os

# ---------------------------------------------------------------------------
# Replace with your token if the repo is private:
#   GITHUB_TOKEN = 'ghp_xxxx'
#   repo_url = f'https://{GITHUB_TOKEN}@github.com/miguelaeh/continual-learning.git'
# ---------------------------------------------------------------------------
repo_url = 'https://github.com/miguelaeh/continual-learning.git'
repo_dir = '/content/continual-learning'

if not os.path.exists(repo_dir):
    !git clone {repo_url} {repo_dir}
else:
    print('Repo already cloned.')

%cd {repo_dir}/6-april-paper
!pip install -e . --quiet
print('Done.')

## Cell 3 — Upload data files\n\nRun this cell and upload these 3 files from your Mac when prompted:\n1. `data/oasst1_train_4000_messages.jsonl` — background stats training data\n2. `data/recovery_eval_sample.jsonl`\n3. `data/recovery_prompts.json`

In [ ]:
from google.colab import files
import os

repo_root = '/content/continual-learning/6-april-paper'
data_dir  = f'{repo_root}/data'
os.makedirs(data_dir, exist_ok=True)

uploads = {
    'oasst1_train_4000_messages.jsonl': data_dir,
    'recovery_eval_sample.jsonl':       data_dir,
    'recovery_prompts.json':            data_dir,
}

for label, dest_dir in uploads.items():
    print(f'Upload {label}')
    up = files.upload()
    for name, data in up.items():
        dest = os.path.join(dest_dir, label)
        open(dest, 'wb').write(data)
        print(f'  Saved → {dest}')

print('All files in place.')

## Cell 4 — Verify files are in place

In [ ]:
import os
repo_root = '/content/continual-learning/6-april-paper'
required = [
    'data/oasst1_train_4000_messages.jsonl',
    'data/recovery_eval_sample.jsonl',
    'data/recovery_prompts.json',
    'experiments/1p5b_kl_n256_v3_collab/recovery.yaml',
    'experiments/1p5b_kl_n256_v3_collab/background.yaml',
    'experiments/1p5b_kl_n256_v3_collab/continual.yaml',
]
all_ok = True
for f in required:
    path = os.path.join(repo_root, f)
    exists = os.path.exists(path)
    size = os.path.getsize(path) if exists else 0
    status = f'OK ({size:,} bytes)' if exists else 'MISSING'
    print(f'  {status:>28}  {f}')
    if not exists:
        all_ok = False
print()
print('Ready to run!' if all_ok else 'Fix missing files above before continuing.')

## Cell 5 — Stage 1: Recovery

In [ ]:
import subprocess, sys

repo_root = '/content/continual-learning/6-april-paper'
EXP = 'experiments/1p5b_kl_n256_v3_collab'

def run(script, config):
    cmd = [sys.executable, script, '--config', config]
    print(f'Running: {" ".join(cmd)}\n')
    proc = subprocess.Popen(cmd, cwd=repo_root, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
    proc.wait()
    print(f'\nExit code: {proc.returncode}')
    if proc.returncode != 0:
        raise RuntimeError(f'{script} failed with exit code {proc.returncode}')

run('scripts/recover.py', f'{EXP}/recovery.yaml')

## Cell 6 — Stage 2: Background stats

In [ ]:
run('scripts/collect_background.py', f'{EXP}/background.yaml')

## Cell 7 — Stage 3: Sparse continual finetuning

In [ ]:
run('scripts/sparse_finetune.py', f'{EXP}/continual.yaml')

## Cell 8 — Download output checkpoints

In [ ]:
from google.colab import files
import os

repo_root  = '/content/continual-learning/6-april-paper'
EXP        = 'experiments/1p5b_kl_n256_v3_collab'

to_download = [
    f'{EXP}/checkpoints/recovery/memory.pt',
    f'{EXP}/checkpoints/background_stats.pt',
    f'{EXP}/checkpoints/continual/memory.pt',
]

for rel in to_download:
    path = os.path.join(repo_root, rel)
    if os.path.exists(path):
        print(f'Downloading {rel}')
        files.download(path)
    else:
        print(f'Not found (stage may not have run): {rel}')